In [ ]:
import pandas as pd
from openai import OpenAI
import os
import re
import json


In [ ]:
# Load the data
df = pd.read_csv("data/transcriptions.csv")
df.head()

In [ ]:

# Initialize the OpenAI client (Make sure your OPENAI_API_KEY environment variable is set)
client = OpenAI()

# 1. Create the sample dataframe
data = {
    'medical_specialty': ['Allergy / Immunology', 'Orthopedic', 'Bariatrics', 'Cardiovascular / Pulmonary', 'Urology'],
    'transcription': [
        'SUBJECTIVE:,  This 23-year-old white female presents...',
        'CHIEF COMPLAINT:,  Achilles ruptured tendon., History: 45-year-old male...',
        'PREOPERATIVE DIAGNOSIS: , Morbid obesity., Patient is a 38-year-old female...',
        'PREOPERATIVE DIAGNOSES, Airway obstruction secondary to severe 60-year-old COPD...',
        'CHIEF COMPLAINT:,  Urinary retention., HISTORY: 72-year-old male...'
    ]
}
df = pd.DataFrame(data)

# 2. Robust function to extract age
def extract_age(text):
    # Searches for patterns like "23-year-old" or "60 year old"
    match = re.search(r'(\d+)\s*-\s*year\s*-\s*old|\d+\s+year\s+old', text, re.IGNORECASE)
    if match:
        # Extract just the digits from the match
        return re.search(r'\d+', match.group()).group()
    return "Unknown"

# 3. Function to recommend treatment using OpenAI
def get_treatment_recommendation(transcription, specialty):
    prompt = f"""
    You are a medical AI assistant. Based on the following medical transcription and specialty,
    provide a brief, 1-sentence treatment recommendation.

    Specialty: {specialty}
    Transcription: {transcription}

    Recommendation:
    """
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini", # Using a fast, cost-effective model for text processing
            messages=[
                {"role": "system", "content": "You are a helpful medical assistant."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=60,
            temperature=0.3
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error generating recommendation: {e}"

# 4. Function to match recommendation with an ICD code (Mock placeholder)
def match_icd_code(recommendation):
    # In production, you would pass this to an embedding model or an external medical coding API.
    # For now, we return a mock placeholder standard.
    return "ICD-10-CM Z00.00"

# 5. Process data and build the new dataframe using a list (much faster than appending to a DF in a loop)
structured_rows = []

for index, row in df.iterrows():
    age = extract_age(row['transcription'])
    specialty = row['medical_specialty'] # Pulling directly from the dataframe's existing column

    # Call OpenAI to get treatment recommendation
    recommendation = get_treatment_recommendation(row['transcription'], specialty)

    # Match recommendation with ICD code
    icd_code = match_icd_code(recommendation)

    # Store the dictionary in our list
    structured_rows.append({
        'age': age,
        'medical_specialty': specialty,
        'recommendation': recommendation,
        'ICD_code': icd_code
    })

# Convert the list of dicts into the final DataFrame
df_structured = pd.DataFrame(structured_rows)

# 6. Print the structured dataframe
print(df_structured)

In [ ]:
import json
import pandas as pd
from openai import OpenAI

# Initialize the OpenAI client
client = OpenAI()

# Load the data
df = pd.read_csv("data/transcriptions.csv")
df.head()

# Define function to extract age and recommended treatment/procedure
def extract_info_with_openai(transcription):
    """Extracts age and recommended treatment from a transcription using OpenAI."""
    messages = [
        {
            "role": "system",
            "content": "You are a healthcare professional extracting patient data. Always return both the age and recommended treatment. If the information is missing, still create the field and specify 'Unknown'.",
            "role": "user",
            "content": f"Please extract and return both the patient's age and recommended treatment from the following transcription. Transcription: {transcription}."
        }
    ]
    function_definition = [
        {
            'type': 'function',
            'function': {
                'name': 'extract_medical_data',
                'description': 'Get the age and recommended treatment from the input text. Always return both age and recommended treatment.',
                'parameters': {
                    'type': 'object',
                    'properties': {
                        'Age': {
                            'type': 'integer',
                            'description': 'Age of the patient'
                        },
                        'Recommended Treatment/Procedure': {
                            'type': 'string',
                            'description': 'Recommended treatment or procedure for the patient'
                        }
                    }
                }
            }
        }
    ]
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=function_definition
    )
    return json.loads(response.choices[0].message.tool_calls[0].function.arguments)

# Define function to extract age and recommended treatment/procedure
def get_icd_codes(treatment):
    if treatment != 'Unknown':
        """Retrieves ICD codes for a given treatment using OpenAI."""
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": f"Provide the ICD codes for the following treatment or procedure: {treatment}. Return the answer as a list of codes. Please only include the codes and no other information."
            }],
            temperature=0.3
        )
        output = response.choices[0].message.content
    else:
        output = 'Unknown'
    return output

# Start an empty list to store processed data
processed_data = []

# Process each row in the DataFrame
for index, row in df.iterrows():
    medical_specialty = row['medical_specialty']
    extracted_data = extract_info_with_openai(row['transcription'])
    icd_code = get_icd_codes(extracted_data["Recommended Treatment/Procedure"]) if 'Recommended Treatment/Procedure' in extracted_data.keys() else 'Unknown'
    extracted_data["Medical Specialty"] = medical_specialty
    extracted_data["ICD Code"] = icd_code

    # Append the extracted information as a new row in the list
    processed_data.append(extracted_data)

# Convert the list to a DataFrame
df_structured = pd.DataFrame(processed_data)